In [0]:
from pyspark.sql.functions import col, to_timestamp, year, month, dayofmonth, dayofweek, hour, minute, when, row_number, to_utc_timestamp, coalesce, lit
from pyspark.sql.window import Window

print("1. Reading Bronze Live Flights...")
bronze_flights = spark.table("aviation_project.bronze_live_flights")

In [0]:
# ---------------------------------------------------------
# Part 1: Local Time Parsing & Feature Extraction
# ---------------------------------------------------------
print("2. Mapping AirLabs data to Local Time & Extracting Features...")

df_local = bronze_flights.withColumn("local_scheduled_time", to_timestamp(col("dep_time"), "yyyy-MM-dd HH:mm"))

df_features = df_local.select(
    col("local_scheduled_time"),
    
    # Temporal Features
    year("local_scheduled_time").alias("year"),
    month("local_scheduled_time").alias("month"),
    dayofmonth("local_scheduled_time").alias("day_of_month"),
    dayofweek("local_scheduled_time").alias("day_of_week"),
    hour("local_scheduled_time").alias("hour"),
    minute("local_scheduled_time").alias("minute"),
    
    # Identifiers
    col("airline_iata").alias("airline_code"),
    col("flight_number").cast("string").alias("flight_number"),
    
    # THE FIX: Inject the missing distance column to match Historical Schema
    lit(None).cast("double").alias("distance_miles"),
    
    # Routing
    col("dep_iata").alias("origin_airport"),
    col("arr_iata").alias("destination_airport"),
    
    # Delay and Status
    coalesce(col("dep_delayed").cast("double").cast("int"), lit(0)).alias("delay_minutes"),
    col("status"), 
    col("bronze_ingested_at")
)

In [0]:
# ---------------------------------------------------------
# Part 2: Timezone Shift & Multi-Class Target
# ---------------------------------------------------------
print("3. Converting to UTC and Calculating Flight Status...")

flight_timezone_expr = when(col("origin_airport").isin("ATL", "CLT", "LGA", "DCA", "MIA"), "America/New_York") \
                .when(col("origin_airport").isin("LIT", "DFW", "ORD", "IAH", "DAL", "STL"), "America/Chicago") \
                .when(col("origin_airport") == "DEN", "America/Denver") \
                .when(col("origin_airport") == "LAS", "America/Los_Angeles") \
                .otherwise("UTC")

df_utc = df_features.withColumn(
    "scheduled_time_utc", 
    to_utc_timestamp(col("local_scheduled_time"), flight_timezone_expr)
).drop("local_scheduled_time")

# Multi-Class Target mapping using the AirLabs status string
df_utc = df_utc.withColumn(
    "flight_status",
    when(col("status") == "cancelled", 2)
    .when(col("status") == "diverted", 3)
    .when(col("delay_minutes") > 15, 1)
    .otherwise(0)
).drop("status")

In [0]:
# ---------------------------------------------------------
# Part 3: The Deduplication Boss Battle
# ---------------------------------------------------------
print("4. Executing Deduplication and Codeshare merging...")

# Group by the physical flight event in absolute UTC time, sort by newest API fetch
window_spec = Window.partitionBy("origin_airport", "destination_airport", "scheduled_time_utc") \
                    .orderBy(col("bronze_ingested_at").desc())

silver_flights = df_utc.withColumn("rn", row_number().over(window_spec)) \
    .filter(col("rn") == 1) \
    .drop("rn", "bronze_ingested_at")

silver_flights = silver_flights.dropna(subset=["scheduled_time_utc"])

In [0]:
display(silver_flights.limit(10))

In [0]:
spark.sql("drop table if exists aviation_project.silver_live_flights")

In [0]:
# ---------------------------------------------------------
# Part 4: Write and Optimize
# ---------------------------------------------------------
print("5. Writing to Silver Delta Table...")
silver_flights.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("aviation_project.silver_live_flights")

print("   -> Optimizing table layout...")
spark.sql("OPTIMIZE aviation_project.silver_live_flights ZORDER BY (scheduled_time_utc)")

final_count = spark.table("aviation_project.silver_live_flights").count()
print(f"6. SUCCESS! Live flights updated and perfectly aligned with Historical schema.")